In [8]:
#imports 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
df=pd.read_csv("movies.csv")
print(df.index)
print("All columns are")
ai=0
for i in df:
    print(ai,i)
    ai+=1

RangeIndex(start=0, stop=9742, step=1)
All columns are
0 Id
1 Title
2 Genres


In [10]:
df.shape

(9742, 3)

In [11]:
df.head()

,Id,Title,Genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Id      9742 non-null   int64 
 1   Title   9742 non-null   object
 2   Genres  9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [13]:
print("In this we don't need to remove empty Genres as we are only using Titles")

In this we don't need to remove empty Genres as we are only using Titles


In [14]:
# Cleaning the movie title of uppercases and years

movies = df

def cleanTitle(title):
    
    title = re.sub(r"\(\d{4}\)", "", title)

   
    title = re.sub(r"[^\w\s]", "", title)

    
    title = title.lower()

   
    title = " ".join(title.split())

    
    return title

movies["CleanTitle"] = movies["Title"].apply(cleanTitle)


In [15]:
movies.head()

,Id,Title,Genres,CleanTitle
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,toy story
1,2,Jumanji (1995),Adventure|Children|Fantasy,jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,grumpier old men
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,waiting to exhale
4,5,Father of the Bride Part II (1995),Comedy,father of the bride part ii


In [16]:
# Using TF-IDF
vectorizer = TfidfVectorizer(stop_words="english")

tfidfMatrix = vectorizer.fit_transform(movies["CleanTitle"])


In [17]:
# Computing cosine similarity

similarityMatrix = cosine_similarity(tfidfMatrix)

In [19]:
# Creatingg similarity DataFrame

similarityDf = pd.DataFrame(
    similarityMatrix,
    index=movies["Title"],
    columns=movies["Title"]
)
similarityDf.head()


Title,Toy Story (1995),Jumanji (1995),Grumpier Old Men (1995),Waiting to Exhale (1995),Father of the Bride Part II (1995),Heat (1995),Sabrina (1995),Tom and Huck (1995),Sudden Death (1995),GoldenEye (1995),...,Gintama: The Movie (2010),anohana: The Flower We Saw That Day - The Movie (2013),Silver Spoon (2014),Love Live! The School Idol Movie (2015),Jon Stewart Has Left the Building (2015),Black Butler: Book of the Atlantic (2017),No Game No Life: Zero (2017),Flint (2017),Bungo Stray Dogs: Dead Apple (2018),Andrew Dice Clay: Dice Rules (1991)
Title,,,,,,,,,,,,,,,,,,,,,
Toy Story (1995),1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Jumanji (1995),0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Grumpier Old Men (1995),0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Waiting to Exhale (1995),0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Father of the Bride Part II (1995),0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
# Our recommendation system

def recommend(movieTitle, topN=10):

    if movieTitle not in similarityDf.index:
        print("Movie not found.")
        return

    recommendations = (
        similarityDf[movieTitle]
        .drop(movieTitle)
        .sort_values(ascending=False)
        .head(topN)
    )

    return recommendations

In [23]:
# Now our function is made let's test it
# Suppose someone just watched Toy Story then what movies should be recommended
print(recommend("Toy Story (1995)"))

Title
Toy Story 3 (2010)           1.000000
Toy Story 2 (1999)           1.000000
Toy, The (1982)              0.823126
Toy Soldiers (1991)          0.596263
Story of Us, The (1999)      0.567859
L.A. Story (1991)            0.411633
Love Story (1970)            0.410634
Christmas Story, A (1983)    0.385169
Ghost Story (1981)           0.364203
True Story (2015)            0.347695
Name: Toy Story (1995), dtype: float64


In [24]:
print(recommend("Home (2015)"))

Title
Home Alone (1990)                        1.000000
Home (2009)                              1.000000
Home Alone 3 (1997)                      1.000000
Last Train Home (2009)                   0.679969
Coming Home (1978)                       0.637514
Stealing Home (1988)                     0.622305
Daddy's Home 2 (2017)                    0.612490
Daddy's Home (2015)                      0.612490
Thin Man Goes Home, The (1945)           0.600736
A Home at the End of the World (2004)    0.591284
Name: Home (2015), dtype: float64


In [25]:
print(recommend("Shrek (2001)"))

Title
Shrek 2 (2004)                                                  1.000000
Shrek the Third (2007)                                          1.000000
Shrek Forever After (a.k.a. Shrek: The Final Chapter) (2010)    0.760493
Shrek the Halls (2007)                                          0.662420
Toy Story (1995)                                                0.000000
Librarian: Return to King Solomon's Mines, The (2006)           0.000000
Librarian: Quest for the Spear, The (2004)                      0.000000
Fay Grim (2006)                                                 0.000000
I'm a Cyborg, But That's OK (Saibogujiman kwenchana) (2006)     0.000000
Breed, The (2006)                                               0.000000
Name: Shrek (2001), dtype: float64
